In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="15yd-u6ChCXVyvkB-u6uFrUJL5daxDfJc", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_01_intro.mp3"))

In [ ]:
# Setup: Run this cell first!
# Standard imports and seed configuration

import random
import json
import time
import os
import tempfile
import copy
import traceback
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Optional, Dict, List, Tuple
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")
print(f"Temp directory: {tempfile.gettempdir()}")
print("All imports loaded successfully.")

# Error Propagation & Crash Recovery

*Part 3 of the Vizuara series on Context Management & Reliability*
*Estimated time: 60 minutes*

## 1. Why Does This Matter?

Multi-agent systems fail silently. This is the single most dangerous property they have.

Consider a research pipeline: Agent A retrieves documents, Agent B extracts facts, Agent C synthesizes a report. Agent B encounters a malformed document and quietly returns an empty list. Agent C receives that empty list, produces a confident-sounding report citing zero evidence, and the user sees a polished output built on nothing.

No error was raised. No warning was shown. The pipeline "succeeded" -- and the output is garbage.

This is not a hypothetical. In production multi-agent systems:

- **Silent failures cascade.** Agent A's partial failure becomes Agent B's corrupted input, which becomes Agent C's hallucinated output. By the time a human sees the result, the root cause is buried three layers deep.
- **Retrying blindly wastes resources.** A permanent failure (invalid API key, missing data source) gets retried 10 times with exponential backoff. Each retry costs tokens and latency, and none of them can succeed.
- **Crashes lose all progress.** A 10-step pipeline crashes at step 8. Without crash recovery, you re-run all 10 steps -- including the 7 that already succeeded and cost real money.

In this notebook, we will build three things that solve these problems:

1. **Structured error propagation** -- every agent returns a typed result (success, partial success, or failure) with enough context to diagnose what went wrong and what still worked
2. **Intelligent retry logic** -- classify failures as transient or permanent, and only retry the ones that have a chance of succeeding
3. **Crash recovery manifests** -- checkpoint progress to disk so that a crashed pipeline can resume from where it left off, not from scratch

By the end, you will have a complete multi-agent pipeline that degrades gracefully, recovers from crashes, and never silently swallows errors.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Let us start with the simplest possible example: a 3-agent pipeline where Agent B partially fails.

We will run the pipeline two ways -- first with silent failure (the default in most implementations), then with structured error propagation -- and compare the outputs.

In [ ]:
# === Silent Failure Mode ===
# This is what most naive multi-agent pipelines look like.
# Errors are swallowed. Downstream agents receive garbage and produce garbage.

def agent_a_retriever(query: str) -> list:
    """Retrieves documents for a query. Always succeeds in this demo."""
    documents = [
        {"id": 1, "text": "Transformers use self-attention to process sequences in parallel."},
        {"id": 2, "text": "The KV cache eliminates redundant computation during inference."},
        {"id": 3, "text": ""},  # <-- Empty document (simulates a retrieval glitch)
        {"id": 4, "text": "Flash Attention reduces memory from O(n^2) to O(n)."},
        {"id": 5, "text": None},  # <-- None (simulates a corrupted record)
    ]
    return documents


def agent_b_extractor_silent(documents: list) -> list:
    """Extracts facts from documents. Silently skips failures."""
    facts = []
    for doc in documents:
        try:
            # Simulate extraction -- just split into sentences
            text = doc["text"]
            if text and len(text) > 10:
                facts.append({"source_id": doc["id"], "fact": text})
            # If text is empty or None, we silently skip it.
            # No error. No warning. No record of what happened.
        except Exception:
            pass  # Swallow the error completely
    return facts


def agent_c_synthesizer(facts: list) -> str:
    """Synthesizes a report from extracted facts."""
    if not facts:
        return "No information available on this topic."
    report = "Research Summary:\n"
    for i, fact in enumerate(facts, 1):
        report += f"  {i}. {fact['fact']} (source: doc {fact['source_id']})\n"
    report += f"\nBased on {len(facts)} sources."
    return report


# Run the silent pipeline
print("=" * 60)
print("SILENT FAILURE MODE")
print("=" * 60)
docs = agent_a_retriever("transformers")
print(f"Retriever returned {len(docs)} documents")

facts = agent_b_extractor_silent(docs)
print(f"Extractor returned {len(facts)} facts (from {len(docs)} docs)")
print(f"  --> {len(docs) - len(facts)} documents silently dropped!")
print(f"  --> The user has NO IDEA that 2 documents failed.\n")

report = agent_c_synthesizer(facts)
print(report)
print("\nProblem: The report looks authoritative but is missing 40% of the data.")
print("The user cannot tell which documents failed or why.")

In [ ]:
# === Structured Error Propagation Mode ===
# Now let us do the same thing, but with structured results.
# Every agent reports exactly what succeeded, what failed, and why.

class ResultStatus(Enum):
    SUCCESS = "success"
    PARTIAL_SUCCESS = "partial_success"
    FAILURE = "failure"


@dataclass
class AgentResult:
    """Structured result from any agent in the pipeline."""
    status: ResultStatus
    data: Any = None
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    metadata: dict = field(default_factory=dict)

    @property
    def succeeded(self) -> bool:
        return self.status in (ResultStatus.SUCCESS, ResultStatus.PARTIAL_SUCCESS)

    @property
    def has_usable_data(self) -> bool:
        return self.data is not None and (not isinstance(self.data, list) or len(self.data) > 0)


def agent_b_extractor_structured(documents: list) -> AgentResult:
    """Extracts facts with full error reporting."""
    facts = []
    errors = []
    total = len(documents)

    for doc in documents:
        doc_id = doc.get("id", "unknown")
        try:
            text = doc.get("text")
            if text is None:
                errors.append({
                    "doc_id": doc_id,
                    "error_type": "null_content",
                    "message": f"Document {doc_id} has null content",
                    "recoverable": True,
                })
                continue
            if len(text.strip()) == 0:
                errors.append({
                    "doc_id": doc_id,
                    "error_type": "empty_content",
                    "message": f"Document {doc_id} has empty content",
                    "recoverable": True,
                })
                continue
            if len(text) > 10:
                facts.append({"source_id": doc_id, "fact": text})
        except Exception as e:
            errors.append({
                "doc_id": doc_id,
                "error_type": type(e).__name__,
                "message": str(e),
                "recoverable": False,
            })

    # Determine status based on results
    if len(facts) == total:
        status = ResultStatus.SUCCESS
    elif len(facts) > 0:
        status = ResultStatus.PARTIAL_SUCCESS
    else:
        status = ResultStatus.FAILURE

    return AgentResult(
        status=status,
        data=facts,
        errors=errors,
        metadata={
            "total_documents": total,
            "successful_extractions": len(facts),
            "failed_extractions": len(errors),
            "success_rate": len(facts) / total if total > 0 else 0,
        },
    )


# Run the structured pipeline
print("=" * 60)
print("STRUCTURED ERROR PROPAGATION MODE")
print("=" * 60)
docs = agent_a_retriever("transformers")
print(f"Retriever returned {len(docs)} documents\n")

result = agent_b_extractor_structured(docs)
print(f"Extractor status: {result.status.value}")
print(f"  Successful: {result.metadata['successful_extractions']}/{result.metadata['total_documents']}")
print(f"  Success rate: {result.metadata['success_rate']:.0%}")
print(f"\nErrors ({len(result.errors)}):")
for err in result.errors:
    print(f"  - Doc {err['doc_id']}: {err['error_type']} -- {err['message']}")
    print(f"    Recoverable: {err['recoverable']}")

# The synthesizer can now make an informed decision
print(f"\nExtractor has usable data: {result.has_usable_data}")
if result.succeeded:
    report = agent_c_synthesizer(result.data)
    if result.status == ResultStatus.PARTIAL_SUCCESS:
        report += f"\n\nWARNING: {result.metadata['failed_extractions']} of "
        report += f"{result.metadata['total_documents']} documents could not be processed."
        report += "\nResults may be incomplete."
    print(f"\n{report}")
else:
    print("\nPipeline aborted: Extractor returned no usable data.")

print("\n" + "=" * 60)
print("Now the user knows EXACTLY what happened:")
print("  - 3 of 5 documents were processed successfully")
print("  - Doc 3 was empty, Doc 5 was null")
print("  - The report is labeled as incomplete")
print("  - Both errors are recoverable (can re-fetch those docs)")

In [ ]:
#@title 🎧 Listen: Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

Let us put numbers on the problem. If each agent in a pipeline has independent probability $p$ of succeeding, what is the probability that the entire pipeline succeeds?

### Pipeline Reliability Without Error Handling

For a pipeline of $n$ agents, each with success probability $p$:

$$P(\text{pipeline success}) = p^n$$

This is brutal. Even with highly reliable agents:

| Agent reliability | 3-agent pipeline | 5-agent pipeline | 10-agent pipeline |
|:-:|:-:|:-:|:-:|
| 99% | 97.0% | 95.1% | 90.4% |
| 95% | 85.7% | 77.4% | 59.9% |
| 90% | 72.9% | 59.0% | 34.9% |

A 10-agent pipeline where each agent is 90% reliable succeeds only 35% of the time. That is worse than a coin flip.

### Improving Effective Reliability

We have three tools to improve this:

**1. Retries for transient failures.** If a failure is transient (network timeout, rate limit) with probability $t$, and permanent with probability $(1-t)$, then $k$ retries give:

$$P(\text{agent success with retries}) = 1 - (1-p)^k \cdot t - (1-p) \cdot (1-t)$$

More simply: if the failure is transient, retrying $k$ times gives success probability $1 - (1-p)^k$. If permanent, retrying never helps.

**2. Partial results.** Instead of binary success/failure, an agent can return partial results. If we define "effective success" as producing $\geq 50\%$ of expected output, the effective reliability jumps dramatically.

**3. Crash recovery.** If the pipeline crashes at step $i$, we only need to re-run steps $i$ through $n$ instead of $1$ through $n$. The expected cost of recovery drops from $O(n)$ to $O(n - i + 1)$.

In [ ]:
# Visualize pipeline reliability with and without error handling

agent_reliabilities = np.linspace(0.80, 0.99, 50)
pipeline_lengths = [3, 5, 10]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Raw pipeline reliability ---
ax = axes[0]
for n in pipeline_lengths:
    pipeline_reliability = agent_reliabilities ** n
    ax.plot(agent_reliabilities * 100, pipeline_reliability * 100,
            linewidth=2, label=f"{n} agents")
ax.set_xlabel("Per-Agent Reliability (%)")
ax.set_ylabel("Pipeline Reliability (%)")
ax.set_title("Pipeline Reliability (No Error Handling)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

# --- Plot 2: Effect of retries (for a 3-agent pipeline) ---
ax = axes[1]
n_agents = 3
transient_fraction = 0.7  # 70% of failures are transient
base_p = 0.90  # 90% per-agent reliability

retry_counts = range(0, 8)
effective_reliabilities = []

for k in retry_counts:
    # Probability of failure after k+1 attempts (initial + k retries)
    # Transient failures: each attempt succeeds independently with probability p
    # Permanent failures: no retry can fix them
    fail_prob = 1 - base_p
    transient_fail = fail_prob * transient_fraction
    permanent_fail = fail_prob * (1 - transient_fraction)

    # After k+1 attempts:
    # - Transient: still failing with prob transient_fail^(k+1) / transient_fraction^k ... 
    # Simpler: P(success) = 1 - P(all attempts fail)
    # P(all fail) = P(permanent) + P(transient) * P(fail k+1 times)
    # For transient: each retry is independent with same p
    p_still_failing = permanent_fail + transient_fail * ((1 - base_p) ** k)
    p_agent = 1 - p_still_failing
    p_pipeline = p_agent ** n_agents
    effective_reliabilities.append(p_pipeline * 100)

ax.bar(list(retry_counts), effective_reliabilities, color='steelblue', alpha=0.8, edgecolor='navy')
ax.axhline(y=base_p**n_agents * 100, color='red', linestyle='--', linewidth=1.5, label=f"No retries: {base_p**n_agents*100:.1f}%")
ax.set_xlabel("Number of Retries")
ax.set_ylabel("Pipeline Reliability (%)")
ax.set_title(f"Effect of Retries ({n_agents}-agent, p={base_p})")
ax.legend()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')

# --- Plot 3: Crash recovery savings ---
ax = axes[2]
n_steps = 10
crash_at = np.arange(1, n_steps + 1)
cost_without_recovery = np.full_like(crash_at, n_steps, dtype=float)
cost_with_recovery = n_steps - crash_at + 1.0

width = 0.35
ax.bar(crash_at - width/2, cost_without_recovery, width, label="No Recovery",
       color='#e74c3c', alpha=0.8, edgecolor='darkred')
ax.bar(crash_at + width/2, cost_with_recovery, width, label="With Recovery",
       color='#2ecc71', alpha=0.8, edgecolor='darkgreen')
ax.set_xlabel("Step Where Crash Occurs")
ax.set_ylabel("Steps to Re-Run")
ax.set_title(f"Crash Recovery Savings ({n_steps}-step pipeline)")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print concrete numbers
print("\nConcrete example: 3-agent pipeline, each 95% reliable")
p = 0.95
print(f"  No error handling:   {p**3 * 100:.1f}% pipeline reliability")
print(f"  With 3 retries:      {(1 - (0.05 * 0.3 + 0.05 * 0.7 * (0.05**3)))**3 * 100:.1f}% pipeline reliability")
print(f"  With partial results: ~95%+ effective reliability (graceful degradation)")

In [ ]:
#@title 🎧 Listen: Structured Errors
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_structured_errors.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It

We will build the four core components of a robust error-handling system, one at a time. Each component addresses a specific failure mode.

### Component 1: Structured Error Result Type

Every agent in our pipeline will return an `AgentResult` with three possible statuses:
- **SUCCESS**: Everything worked. Full output available.
- **PARTIAL_SUCCESS**: Some items failed, but we have usable partial output.
- **FAILURE**: Nothing worked. No usable output.

We already saw the basic version above. Now let us make it production-grade.

In [ ]:
# ============================================================
# Component 1: Production-grade structured error result type
# ============================================================

class ErrorSeverity(Enum):
    """How severe is this error?"""
    WARNING = "warning"      # Non-blocking: degraded quality but usable
    ERROR = "error"          # Blocking for this item, but pipeline can continue
    CRITICAL = "critical"    # Pipeline must stop


@dataclass
class StructuredError:
    """A single error with full diagnostic context."""
    error_type: str            # e.g., "timeout", "parse_error", "auth_failure"
    message: str               # Human-readable description
    severity: ErrorSeverity    # How bad is it?
    is_transient: bool         # Can a retry fix this?
    source_agent: str          # Which agent produced this error?
    item_id: Optional[str] = None  # Which input item caused the error?
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    retry_count: int = 0       # How many times has this been retried?

    def to_dict(self) -> dict:
        return {
            "error_type": self.error_type,
            "message": self.message,
            "severity": self.severity.value,
            "is_transient": self.is_transient,
            "source_agent": self.source_agent,
            "item_id": self.item_id,
            "timestamp": self.timestamp,
            "retry_count": self.retry_count,
        }


@dataclass
class PipelineResult:
    """Result from any agent in the pipeline."""
    status: ResultStatus
    agent_name: str
    data: Any = None
    errors: List[StructuredError] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    duration_seconds: float = 0.0

    @property
    def succeeded(self) -> bool:
        return self.status in (ResultStatus.SUCCESS, ResultStatus.PARTIAL_SUCCESS)

    @property
    def has_usable_data(self) -> bool:
        if self.data is None:
            return False
        if isinstance(self.data, (list, dict)) and len(self.data) == 0:
            return False
        return True

    @property
    def critical_errors(self) -> List[StructuredError]:
        return [e for e in self.errors if e.severity == ErrorSeverity.CRITICAL]

    @property
    def transient_errors(self) -> List[StructuredError]:
        return [e for e in self.errors if e.is_transient]

    @property
    def permanent_errors(self) -> List[StructuredError]:
        return [e for e in self.errors if not e.is_transient]

    def summary(self) -> str:
        lines = [f"[{self.agent_name}] Status: {self.status.value} ({self.duration_seconds:.2f}s)"]
        if self.errors:
            lines.append(f"  Errors: {len(self.errors)} ({len(self.transient_errors)} transient, {len(self.permanent_errors)} permanent)")
            for err in self.errors:
                lines.append(f"    - [{err.severity.value}] {err.error_type}: {err.message}")
        if self.metadata:
            for k, v in self.metadata.items():
                lines.append(f"  {k}: {v}")
        return "\n".join(lines)

    def to_dict(self) -> dict:
        return {
            "status": self.status.value,
            "agent_name": self.agent_name,
            "errors": [e.to_dict() for e in self.errors],
            "metadata": self.metadata,
            "duration_seconds": self.duration_seconds,
        }


# Demonstrate the structured result
result = PipelineResult(
    status=ResultStatus.PARTIAL_SUCCESS,
    agent_name="FactExtractor",
    data=[{"fact": "Transformers use self-attention"}],
    errors=[
        StructuredError(
            error_type="timeout",
            message="Document 3 retrieval timed out after 30s",
            severity=ErrorSeverity.ERROR,
            is_transient=True,
            source_agent="FactExtractor",
            item_id="doc_3",
        ),
        StructuredError(
            error_type="parse_error",
            message="Document 5 contains invalid UTF-8",
            severity=ErrorSeverity.ERROR,
            is_transient=False,
            source_agent="FactExtractor",
            item_id="doc_5",
        ),
    ],
    metadata={"total_items": 5, "successful": 3, "failed": 2},
    duration_seconds=2.4,
)

print(result.summary())
print(f"\nHas usable data: {result.has_usable_data}")
print(f"Critical errors: {len(result.critical_errors)}")
print(f"Transient (retryable): {len(result.transient_errors)}")
print(f"Permanent (not retryable): {len(result.permanent_errors)}")

In [ ]:
#@title 🎧 Listen: Retry Logic
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_retry_logic.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Component 2: Transient vs Permanent Failure Classifier with Retry Logic

Not all failures are created equal. A network timeout might succeed on the next attempt. An invalid API key will fail forever.

Our retry logic classifies failures and only retries the ones that have a chance of succeeding.

In [ ]:
# ============================================================
# Component 2: Failure classifier + retry logic
# ============================================================

# Failure classification rules
TRANSIENT_ERROR_TYPES = {
    "timeout", "rate_limit", "connection_reset", "server_error",
    "temporary_unavailable", "network_error",
}

PERMANENT_ERROR_TYPES = {
    "auth_failure", "invalid_input", "parse_error", "not_found",
    "permission_denied", "schema_violation", "data_corruption",
}


def classify_error(error_type: str) -> bool:
    """Returns True if the error is transient (retryable)."""
    if error_type in TRANSIENT_ERROR_TYPES:
        return True
    if error_type in PERMANENT_ERROR_TYPES:
        return False
    # Unknown errors: assume transient for safety (will be retried once)
    return True


def retry_with_classification(
    func,
    args: tuple = (),
    kwargs: dict = None,
    max_retries: int = 3,
    base_delay: float = 0.1,
    agent_name: str = "Agent",
) -> PipelineResult:
    """
    Execute a function with intelligent retry logic.
    
    - Classifies errors as transient or permanent
    - Only retries transient failures
    - Immediately gives up on permanent failures
    - Returns a PipelineResult with full error context
    """
    kwargs = kwargs or {}
    all_errors = []
    attempt = 0
    start_time = time.time()

    while attempt <= max_retries:
        try:
            result = func(*args, **kwargs)
            duration = time.time() - start_time

            # If the function returns a PipelineResult, augment it
            if isinstance(result, PipelineResult):
                result.duration_seconds = duration
                result.errors = all_errors + result.errors
                if attempt > 0:
                    result.metadata["retries_needed"] = attempt
                return result

            # Otherwise wrap in a PipelineResult
            return PipelineResult(
                status=ResultStatus.SUCCESS,
                agent_name=agent_name,
                data=result,
                errors=all_errors,
                metadata={"retries_needed": attempt} if attempt > 0 else {},
                duration_seconds=duration,
            )

        except Exception as e:
            error_type = type(e).__name__.lower()
            is_transient = classify_error(error_type)

            structured_err = StructuredError(
                error_type=error_type,
                message=str(e),
                severity=ErrorSeverity.ERROR if is_transient else ErrorSeverity.CRITICAL,
                is_transient=is_transient,
                source_agent=agent_name,
                retry_count=attempt,
            )
            all_errors.append(structured_err)

            if not is_transient:
                print(f"  [{agent_name}] Permanent failure: {error_type} -- not retrying")
                break

            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)  # Exponential backoff
                print(f"  [{agent_name}] Transient failure: {error_type} -- retrying in {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                print(f"  [{agent_name}] Max retries ({max_retries}) exhausted")

            attempt += 1

    duration = time.time() - start_time
    return PipelineResult(
        status=ResultStatus.FAILURE,
        agent_name=agent_name,
        data=None,
        errors=all_errors,
        metadata={"retries_attempted": attempt},
        duration_seconds=duration,
    )


# --- Demo: Transient failure that recovers ---
call_count = 0

class Timeout(Exception):
    pass

class AuthFailure(Exception):
    pass

def flaky_function():
    """Fails twice with timeout, then succeeds."""
    global call_count
    call_count += 1
    if call_count <= 2:
        raise Timeout(f"Connection timed out (attempt {call_count})")
    return {"result": "data fetched successfully"}

print("--- Transient failure (recovers after 2 retries) ---")
call_count = 0
result = retry_with_classification(flaky_function, agent_name="DataFetcher")
print(f"\n{result.summary()}")

# --- Demo: Permanent failure (stops immediately) ---
def broken_function():
    raise AuthFailure("Invalid API key: sk-xxx...xxx")

print("\n--- Permanent failure (stops immediately) ---")
result = retry_with_classification(broken_function, agent_name="APIClient")
print(f"\n{result.summary()}")

In [ ]:
#@title 🎧 Listen: Crash Recovery
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_crash_recovery.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Component 3: Crash Recovery Manifest

A crash recovery manifest is a JSON file that records the pipeline's progress after each step. If the process crashes (or the notebook kernel restarts), we can read the manifest and resume from the last completed step.

In [ ]:
# ============================================================
# Component 3: Crash Recovery Manifest
# ============================================================

@dataclass
class StepCheckpoint:
    """A single pipeline step's checkpoint."""
    step_name: str
    step_index: int
    status: str       # "completed", "failed", "pending"
    output_data: Any = None
    errors: list = field(default_factory=list)
    started_at: str = ""
    completed_at: str = ""


class CrashRecoveryManifest:
    """
    Saves pipeline progress to disk after each step.
    On restart, loads the manifest and resumes from the last completed step.
    """

    def __init__(self, pipeline_id: str, manifest_dir: Optional[str] = None):
        self.pipeline_id = pipeline_id
        self.manifest_dir = manifest_dir or tempfile.mkdtemp(prefix="pipeline_")
        self.manifest_path = os.path.join(self.manifest_dir, f"{pipeline_id}_manifest.json")
        self.checkpoints: List[StepCheckpoint] = []
        self.created_at = datetime.now().isoformat()
        self.last_updated = self.created_at

    def _save(self):
        """Persist the manifest to disk."""
        data = {
            "pipeline_id": self.pipeline_id,
            "created_at": self.created_at,
            "last_updated": datetime.now().isoformat(),
            "total_steps": len(self.checkpoints),
            "completed_steps": sum(1 for c in self.checkpoints if c.status == "completed"),
            "checkpoints": [
                {
                    "step_name": c.step_name,
                    "step_index": c.step_index,
                    "status": c.status,
                    "output_data": c.output_data,
                    "errors": c.errors,
                    "started_at": c.started_at,
                    "completed_at": c.completed_at,
                }
                for c in self.checkpoints
            ],
        }
        with open(self.manifest_path, 'w') as f:
            json.dump(data, f, indent=2, default=str)

    @classmethod
    def load(cls, manifest_path: str) -> 'CrashRecoveryManifest':
        """Load a manifest from disk (after a crash)."""
        with open(manifest_path, 'r') as f:
            data = json.load(f)

        manifest = cls(
            pipeline_id=data["pipeline_id"],
            manifest_dir=os.path.dirname(manifest_path),
        )
        manifest.created_at = data["created_at"]
        manifest.checkpoints = [
            StepCheckpoint(
                step_name=c["step_name"],
                step_index=c["step_index"],
                status=c["status"],
                output_data=c["output_data"],
                errors=c["errors"],
                started_at=c["started_at"],
                completed_at=c["completed_at"],
            )
            for c in data["checkpoints"]
        ]
        return manifest

    def register_steps(self, step_names: List[str]):
        """Register all steps upfront."""
        self.checkpoints = [
            StepCheckpoint(step_name=name, step_index=i, status="pending")
            for i, name in enumerate(step_names)
        ]
        self._save()

    def mark_started(self, step_index: int):
        """Mark a step as started."""
        self.checkpoints[step_index].status = "running"
        self.checkpoints[step_index].started_at = datetime.now().isoformat()
        self._save()

    def mark_completed(self, step_index: int, output_data: Any = None):
        """Mark a step as completed and save its output."""
        self.checkpoints[step_index].status = "completed"
        self.checkpoints[step_index].output_data = output_data
        self.checkpoints[step_index].completed_at = datetime.now().isoformat()
        self._save()

    def mark_failed(self, step_index: int, errors: list = None):
        """Mark a step as failed."""
        self.checkpoints[step_index].status = "failed"
        self.checkpoints[step_index].errors = errors or []
        self.checkpoints[step_index].completed_at = datetime.now().isoformat()
        self._save()

    def get_resume_point(self) -> int:
        """Find the first non-completed step to resume from."""
        for cp in self.checkpoints:
            if cp.status != "completed":
                return cp.step_index
        return len(self.checkpoints)  # All done

    def get_step_output(self, step_index: int) -> Any:
        """Retrieve the saved output of a completed step."""
        if step_index < len(self.checkpoints) and self.checkpoints[step_index].status == "completed":
            return self.checkpoints[step_index].output_data
        return None

    def summary(self) -> str:
        lines = [f"Pipeline: {self.pipeline_id}"]
        lines.append(f"Manifest: {self.manifest_path}")
        for cp in self.checkpoints:
            status_icon = {"completed": "[done]", "failed": "[FAIL]", "pending": "[    ]", "running": "[ >> ]"}[cp.status]
            lines.append(f"  {status_icon} Step {cp.step_index}: {cp.step_name}")
        completed = sum(1 for c in self.checkpoints if c.status == "completed")
        lines.append(f"Progress: {completed}/{len(self.checkpoints)} steps completed")
        return "\n".join(lines)


# --- Demo: Create a manifest, simulate partial progress, then recover ---
print("=== Creating a new pipeline manifest ===")
manifest = CrashRecoveryManifest("research_pipeline_001")
manifest.register_steps(["retrieve_documents", "extract_facts", "synthesize_report"])
print(manifest.summary())

print("\n=== Simulating step 0 completion ===")
manifest.mark_started(0)
manifest.mark_completed(0, output_data={"documents": ["doc1", "doc2", "doc3"]})
print(manifest.summary())

print("\n=== Simulating crash during step 1 ===")
manifest.mark_started(1)
# CRASH! (we just stop here -- no mark_completed)
print(manifest.summary())

print("\n=== Recovering from crash ===")
recovered = CrashRecoveryManifest.load(manifest.manifest_path)
resume_at = recovered.get_resume_point()
print(f"Resume point: step {resume_at} ({recovered.checkpoints[resume_at].step_name})")
print(f"Step 0 output (cached): {recovered.get_step_output(0)}")
print("\nWe skip step 0 entirely -- its output is already saved!")

In [ ]:
#@title 🎧 Listen: Graceful Degradation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_graceful_degradation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Component 4: Graceful Degradation with Partial Results

The final component ties everything together. Instead of failing completely when one item in a batch fails, we return whatever we successfully processed and annotate the gaps.

In [ ]:
# ============================================================
# Component 4: Graceful degradation -- partial result aggregator
# ============================================================

class GracefulExecutor:
    """
    Executes a function over a batch of items, collecting partial results.
    If some items fail, returns the successful ones instead of failing entirely.
    """

    def __init__(self, agent_name: str, max_retries: int = 2, base_delay: float = 0.05):
        self.agent_name = agent_name
        self.max_retries = max_retries
        self.base_delay = base_delay

    def execute_batch(
        self,
        func,
        items: list,
        item_id_fn = None,
    ) -> PipelineResult:
        """
        Apply func to each item in the batch.
        Collect successes and failures separately.
        Return partial results if some items fail.
        """
        if item_id_fn is None:
            item_id_fn = lambda i, item: str(i)

        start_time = time.time()
        successes = []
        all_errors = []

        for i, item in enumerate(items):
            item_id = item_id_fn(i, item)
            attempt = 0
            succeeded = False

            while attempt <= self.max_retries:
                try:
                    result = func(item)
                    successes.append({"item_id": item_id, "result": result})
                    succeeded = True
                    break
                except Exception as e:
                    error_type = type(e).__name__.lower()
                    is_transient = classify_error(error_type)

                    err = StructuredError(
                        error_type=error_type,
                        message=str(e),
                        severity=ErrorSeverity.ERROR,
                        is_transient=is_transient,
                        source_agent=self.agent_name,
                        item_id=item_id,
                        retry_count=attempt,
                    )

                    if not is_transient:
                        all_errors.append(err)
                        break  # Don't retry permanent failures

                    if attempt < self.max_retries:
                        delay = self.base_delay * (2 ** attempt)
                        time.sleep(delay)
                    else:
                        all_errors.append(err)

                    attempt += 1

        # Determine overall status
        duration = time.time() - start_time
        total = len(items)
        n_success = len(successes)

        if n_success == total:
            status = ResultStatus.SUCCESS
        elif n_success > 0:
            status = ResultStatus.PARTIAL_SUCCESS
        else:
            status = ResultStatus.FAILURE

        return PipelineResult(
            status=status,
            agent_name=self.agent_name,
            data=[s["result"] for s in successes],
            errors=all_errors,
            metadata={
                "total_items": total,
                "successful": n_success,
                "failed": total - n_success,
                "success_rate": n_success / total if total > 0 else 0,
            },
            duration_seconds=duration,
        )


# --- Demo: Process a batch of 10 items where 3 will fail ---
random.seed(42)

class Timeout(Exception):
    pass

class DataCorruption(Exception):
    pass


def process_document(doc: dict) -> dict:
    """Simulate processing a document. Some will fail."""
    # Simulate random failures
    roll = random.random()
    if roll < 0.15:  # 15% transient failure
        raise Timeout(f"Processing document '{doc['title']}' timed out")
    if roll < 0.20:  # 5% permanent failure
        raise DataCorruption(f"Document '{doc['title']}' has corrupted data")

    return {
        "title": doc["title"],
        "summary": f"Summary of: {doc['title']}",
        "word_count": random.randint(100, 1000),
    }


documents = [{"title": f"Research Paper {i+1}", "id": i} for i in range(10)]

executor = GracefulExecutor("DocumentProcessor", max_retries=2, base_delay=0.01)
result = executor.execute_batch(
    process_document,
    documents,
    item_id_fn=lambda i, doc: f"doc_{doc['id']}",
)

print(result.summary())
print(f"\nUsable results: {len(result.data)} of {result.metadata['total_items']}")
if result.data:
    print("\nSuccessful results:")
    for item in result.data[:3]:
        print(f"  - {item['title']}: {item['word_count']} words")
    if len(result.data) > 3:
        print(f"  ... and {len(result.data) - 3} more")

In [ ]:
#@title 🎧 Listen: Exercise Backoff
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_exercise_backoff.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### Exercise 1: Add Exponential Backoff with Jitter

Our retry logic uses simple exponential backoff: `delay = base_delay * (2 ** attempt)`. In production, you should add **jitter** (randomness) to prevent the "thundering herd" problem -- where many clients retry at exactly the same time.

Your task: implement the `compute_backoff_delay` function that uses exponential backoff with full jitter:

$$\text{delay} = \text{random}(0, \min(\text{cap}, \text{base} \times 2^{\text{attempt}}))$$

In [ ]:
# Exercise 1: Implement exponential backoff with jitter

def compute_backoff_delay(
    attempt: int,
    base_delay: float = 0.1,
    max_delay: float = 30.0,
) -> float:
    """
    Compute the delay before the next retry attempt.
    
    Uses exponential backoff with full jitter:
      delay = random(0, min(max_delay, base_delay * 2^attempt))
    
    Args:
        attempt: The retry attempt number (0-indexed)
        base_delay: Base delay in seconds
        max_delay: Maximum delay cap in seconds
    
    Returns:
        Delay in seconds (float)
    """
    # TODO: Implement this function.
    # 1. Calculate the exponential delay: base_delay * (2 ** attempt)
    # 2. Cap it at max_delay using min()
    # 3. Add jitter: pick a random value between 0 and the capped delay
    # Hint: use random.uniform(0, capped_delay)

    pass  # YOUR CODE HERE


# Test your implementation:
# Uncomment the lines below after implementing

# random.seed(42)
# print("Attempt | Exponential Delay | With Jitter")
# print("-" * 45)
# for attempt in range(8):
#     jittered = compute_backoff_delay(attempt, base_delay=0.1, max_delay=30.0)
#     raw = min(30.0, 0.1 * (2 ** attempt))
#     print(f"   {attempt}    |      {raw:6.2f}s      |   {jittered:.3f}s")
#
# # Verify: jittered delay should always be <= raw exponential delay
# # Verify: jittered delay should always be >= 0
# # Verify: delay should be capped at max_delay

In [ ]:
#@title 🎧 Listen: Exercise Crash Recovery
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_exercise_crash_recovery.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Exercise 2: Implement Crash Recovery Resume

You have a 5-step pipeline. It crashes at step 3. Your task: implement the `resume_pipeline` function that:
1. Loads the crash recovery manifest
2. Determines which step to resume from
3. Retrieves cached outputs from completed steps
4. Runs only the remaining steps

In [ ]:
# Exercise 2: Implement crash recovery resume

# First, let us set up a pipeline that crashes partway through
STEP_FUNCTIONS = {
    "step_a_collect": lambda data: {"collected": [1, 2, 3, 4, 5]},
    "step_b_filter": lambda data: {"filtered": [x for x in data["collected"] if x > 2]},
    "step_c_transform": lambda data: {"transformed": [x * 10 for x in data["filtered"]]},
    "step_d_aggregate": lambda data: {"total": sum(data["transformed"])},
    "step_e_format": lambda data: {"report": f"Final result: {data['total']}"},
}


def run_pipeline_with_crash(crash_at_step: int = 3):
    """Run the pipeline but simulate a crash at the given step."""
    manifest = CrashRecoveryManifest("exercise_pipeline")
    step_names = list(STEP_FUNCTIONS.keys())
    manifest.register_steps(step_names)

    current_data = {}
    for i, (name, func) in enumerate(STEP_FUNCTIONS.items()):
        if i == crash_at_step:
            manifest.mark_started(i)
            print(f"  CRASH at step {i} ({name})!")
            return manifest.manifest_path  # Return path for recovery

        manifest.mark_started(i)
        current_data = func(current_data)
        manifest.mark_completed(i, output_data=current_data)
        print(f"  Completed step {i}: {name} -> {current_data}")

    return manifest.manifest_path


def resume_pipeline(manifest_path: str):
    """
    Resume a crashed pipeline from its manifest.
    
    TODO: Implement this function.
    
    Steps:
    1. Load the manifest from manifest_path using CrashRecoveryManifest.load()
    2. Get the resume point using manifest.get_resume_point()
    3. Get the output from the last completed step (resume_point - 1)
       using manifest.get_step_output()
    4. Run the remaining steps starting from the resume point
    5. Mark each step as started and completed in the manifest
    6. Return the final output
    """
    # YOUR CODE HERE
    pass


# Run the pipeline (it will crash at step 3)
print("=== Initial pipeline run (crashes at step 3) ===")
manifest_path = run_pipeline_with_crash(crash_at_step=3)

# TODO: Uncomment the lines below after implementing resume_pipeline
# print("\n=== Resuming from crash ===")
# final_output = resume_pipeline(manifest_path)
# print(f"\nFinal output: {final_output}")
# print("Pipeline completed successfully after recovery!")

In [ ]:
#@title 🎧 Listen: Full Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_full_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Now we build a complete multi-agent research pipeline that combines all four components:
- **Structured error results** at every stage
- **Failure classification** with intelligent retries
- **Crash recovery manifests** for resumability
- **Graceful degradation** with partial results

The pipeline: **Document Retriever** --> **Fact Extractor** --> **Report Synthesizer**

Each agent simulates realistic work and realistic failures.

In [ ]:
# ============================================================
# Complete Multi-Agent Research Pipeline
# ============================================================

# Simulated document database
DOCUMENT_DB = {
    f"doc_{i}": {
        "id": f"doc_{i}",
        "title": title,
        "content": content,
    }
    for i, (title, content) in enumerate([
        ("Attention Is All You Need", "Transformers replace recurrence with self-attention. Multi-head attention allows attending to different representation subspaces. Positional encoding provides sequence order information."),
        ("BERT: Pre-training of Deep Bidirectional Transformers", "Masked language modeling enables bidirectional pre-training. Fine-tuning on downstream tasks achieves state-of-the-art. WordPiece tokenization handles rare words."),
        ("GPT-3: Language Models are Few-Shot Learners", "Scaling to 175B parameters enables in-context learning. Few-shot prompting reduces need for task-specific fine-tuning. Emergent abilities appear at scale."),
        ("Scaling Laws for Neural Language Models", "Loss follows power law with model size, dataset size, and compute. Optimal allocation follows Chinchilla scaling. Larger models are more sample efficient."),
        ("Flash Attention: Fast and Memory-Efficient Exact Attention", "IO-aware attention algorithm reduces memory from O(n^2) to O(n). Kernel fusion avoids materializing the attention matrix. Enables longer context lengths."),
        ("Chain-of-Thought Prompting", "Step-by-step reasoning improves accuracy on complex tasks. Emerges at sufficient model scale. Zero-shot CoT works with 'let us think step by step'."),
        ("Constitutional AI", "Self-supervised critique and revision improves safety. RLHF with AI feedback reduces human annotation needs. Principles guide model behavior."),
        ("Retrieval-Augmented Generation", "Combining retrieval with generation grounds responses in evidence. Dense passage retrieval outperforms sparse methods. RAG reduces hallucination in knowledge-intensive tasks."),
    ])
}


class DocumentRetriever:
    """Agent 1: Retrieves relevant documents for a query."""

    def __init__(self, failure_rate: float = 0.10):
        self.failure_rate = failure_rate
        self.name = "DocumentRetriever"

    def retrieve(self, query: str, n_docs: int = 5) -> PipelineResult:
        start = time.time()
        all_docs = list(DOCUMENT_DB.values())
        random.shuffle(all_docs)
        selected = all_docs[:n_docs]
        retrieved = []
        errors = []

        for doc in selected:
            roll = random.random()
            if roll < self.failure_rate * 0.7:  # Transient
                errors.append(StructuredError(
                    error_type="timeout",
                    message=f"Timeout retrieving '{doc['title']}'",
                    severity=ErrorSeverity.ERROR,
                    is_transient=True,
                    source_agent=self.name,
                    item_id=doc["id"],
                ))
            elif roll < self.failure_rate:  # Permanent
                errors.append(StructuredError(
                    error_type="not_found",
                    message=f"Document '{doc['title']}' not found in index",
                    severity=ErrorSeverity.ERROR,
                    is_transient=False,
                    source_agent=self.name,
                    item_id=doc["id"],
                ))
            else:
                retrieved.append(doc)

        n_total = len(selected)
        n_ok = len(retrieved)
        status = ResultStatus.SUCCESS if n_ok == n_total else (
            ResultStatus.PARTIAL_SUCCESS if n_ok > 0 else ResultStatus.FAILURE
        )

        return PipelineResult(
            status=status,
            agent_name=self.name,
            data=retrieved,
            errors=errors,
            metadata={"requested": n_docs, "retrieved": n_ok, "failed": n_total - n_ok},
            duration_seconds=time.time() - start,
        )


class FactExtractor:
    """Agent 2: Extracts structured facts from documents."""

    def __init__(self, failure_rate: float = 0.10):
        self.failure_rate = failure_rate
        self.name = "FactExtractor"

    def extract(self, documents: list) -> PipelineResult:
        start = time.time()
        all_facts = []
        errors = []

        for doc in documents:
            roll = random.random()
            if roll < self.failure_rate * 0.7:
                errors.append(StructuredError(
                    error_type="timeout",
                    message=f"Extraction timed out for '{doc['title']}'",
                    severity=ErrorSeverity.ERROR,
                    is_transient=True,
                    source_agent=self.name,
                    item_id=doc["id"],
                ))
                continue
            elif roll < self.failure_rate:
                errors.append(StructuredError(
                    error_type="parse_error",
                    message=f"Could not parse content of '{doc['title']}'",
                    severity=ErrorSeverity.ERROR,
                    is_transient=False,
                    source_agent=self.name,
                    item_id=doc["id"],
                ))
                continue

            # "Extract" facts by splitting content into sentences
            sentences = [s.strip() for s in doc["content"].split(".") if s.strip()]
            for sent in sentences:
                all_facts.append({
                    "source_id": doc["id"],
                    "source_title": doc["title"],
                    "fact": sent + ".",
                    "confidence": round(random.uniform(0.75, 0.99), 2),
                })

        n_docs = len(documents)
        n_docs_ok = n_docs - len(errors)
        status = ResultStatus.SUCCESS if n_docs_ok == n_docs else (
            ResultStatus.PARTIAL_SUCCESS if n_docs_ok > 0 else ResultStatus.FAILURE
        )

        return PipelineResult(
            status=status,
            agent_name=self.name,
            data=all_facts,
            errors=errors,
            metadata={
                "docs_processed": n_docs_ok,
                "docs_failed": len(errors),
                "facts_extracted": len(all_facts),
            },
            duration_seconds=time.time() - start,
        )


class ReportSynthesizer:
    """Agent 3: Synthesizes a research report from extracted facts."""

    def __init__(self, failure_rate: float = 0.05):
        self.failure_rate = failure_rate
        self.name = "ReportSynthesizer"

    def synthesize(self, facts: list, query: str) -> PipelineResult:
        start = time.time()

        if random.random() < self.failure_rate:
            return PipelineResult(
                status=ResultStatus.FAILURE,
                agent_name=self.name,
                data=None,
                errors=[StructuredError(
                    error_type="server_error",
                    message="Synthesis service temporarily unavailable",
                    severity=ErrorSeverity.CRITICAL,
                    is_transient=True,
                    source_agent=self.name,
                )],
                duration_seconds=time.time() - start,
            )

        # Group facts by source
        source_groups = {}
        for fact in facts:
            title = fact["source_title"]
            if title not in source_groups:
                source_groups[title] = []
            source_groups[title].append(fact)

        # Build report
        report_lines = [f"Research Report: {query}", "=" * 50, ""]
        for title, group_facts in source_groups.items():
            report_lines.append(f"From '{title}':")
            for f in group_facts:
                report_lines.append(f"  - {f['fact']} (confidence: {f['confidence']})")
            report_lines.append("")

        report_lines.append(f"Total facts: {len(facts)} from {len(source_groups)} sources")
        avg_conf = np.mean([f["confidence"] for f in facts]) if facts else 0
        report_lines.append(f"Average confidence: {avg_conf:.2f}")

        report = "\n".join(report_lines)

        return PipelineResult(
            status=ResultStatus.SUCCESS,
            agent_name=self.name,
            data=report,
            metadata={
                "facts_used": len(facts),
                "sources_cited": len(source_groups),
                "avg_confidence": round(avg_conf, 2),
            },
            duration_seconds=time.time() - start,
        )


print("Agents defined: DocumentRetriever, FactExtractor, ReportSynthesizer")
print(f"Document database: {len(DOCUMENT_DB)} papers")

In [ ]:
# ============================================================
# The Full Pipeline Orchestrator
# ============================================================

class ResearchPipeline:
    """
    Orchestrates the 3-agent pipeline with:
    - Structured error propagation between agents
    - Crash recovery manifests
    - Graceful degradation at every stage
    - Configurable retry logic
    """

    def __init__(
        self,
        pipeline_id: str = None,
        retriever_failure_rate: float = 0.10,
        extractor_failure_rate: float = 0.10,
        synthesizer_failure_rate: float = 0.05,
        max_retries: int = 2,
    ):
        self.pipeline_id = pipeline_id or f"pipeline_{int(time.time())}_{random.randint(1000,9999)}"
        self.retriever = DocumentRetriever(retriever_failure_rate)
        self.extractor = FactExtractor(extractor_failure_rate)
        self.synthesizer = ReportSynthesizer(synthesizer_failure_rate)
        self.max_retries = max_retries
        self.step_results: List[PipelineResult] = []

    def run(self, query: str, n_docs: int = 5, manifest_dir: str = None) -> dict:
        """
        Run the full pipeline with error handling and crash recovery.
        Returns a dict with the final report and all metadata.
        """
        manifest = CrashRecoveryManifest(self.pipeline_id, manifest_dir)
        manifest.register_steps(["retrieve", "extract", "synthesize"])
        self.step_results = []

        all_errors = []

        # --- Step 1: Retrieve Documents ---
        manifest.mark_started(0)
        retrieve_result = self._retry_step(
            lambda: self.retriever.retrieve(query, n_docs),
            "DocumentRetriever",
        )
        self.step_results.append(retrieve_result)
        all_errors.extend(retrieve_result.errors)

        if not retrieve_result.has_usable_data:
            manifest.mark_failed(0, [e.to_dict() for e in retrieve_result.errors])
            return self._build_output("FAILED", None, all_errors, query, manifest)

        manifest.mark_completed(0, {"n_docs": len(retrieve_result.data)})

        # --- Step 2: Extract Facts ---
        manifest.mark_started(1)
        extract_result = self._retry_step(
            lambda: self.extractor.extract(retrieve_result.data),
            "FactExtractor",
        )
        self.step_results.append(extract_result)
        all_errors.extend(extract_result.errors)

        if not extract_result.has_usable_data:
            manifest.mark_failed(1, [e.to_dict() for e in extract_result.errors])
            return self._build_output("FAILED", None, all_errors, query, manifest)

        manifest.mark_completed(1, {"n_facts": len(extract_result.data)})

        # --- Step 3: Synthesize Report ---
        manifest.mark_started(2)
        synth_result = self._retry_step(
            lambda: self.synthesizer.synthesize(extract_result.data, query),
            "ReportSynthesizer",
        )
        self.step_results.append(synth_result)
        all_errors.extend(synth_result.errors)

        if not synth_result.has_usable_data:
            manifest.mark_failed(2, [e.to_dict() for e in synth_result.errors])
            # Graceful degradation: return raw facts if synthesis fails
            fallback_report = self._build_fallback_report(extract_result.data, query)
            return self._build_output("DEGRADED", fallback_report, all_errors, query, manifest)

        manifest.mark_completed(2)

        # Determine overall status
        any_partial = any(r.status == ResultStatus.PARTIAL_SUCCESS for r in self.step_results)
        overall = "PARTIAL" if any_partial else "SUCCESS"

        return self._build_output(overall, synth_result.data, all_errors, query, manifest)

    def _retry_step(self, step_fn, agent_name: str) -> PipelineResult:
        """Retry a pipeline step, respecting failure classification."""
        best_result = None
        for attempt in range(self.max_retries + 1):
            result = step_fn()
            if result.succeeded:
                if attempt > 0:
                    result.metadata["retries_needed"] = attempt
                return result

            best_result = result

            # Check if any errors are permanent (no point retrying)
            if all(not e.is_transient for e in result.errors):
                break

            if attempt < self.max_retries:
                delay = 0.01 * (2 ** attempt)
                time.sleep(delay)

        return best_result

    def _build_fallback_report(self, facts: list, query: str) -> str:
        """Build a minimal report from raw facts when synthesis fails."""
        lines = [f"[DEGRADED] Research Summary: {query}", ""]
        lines.append("Note: Report synthesis failed. Showing raw extracted facts.")
        lines.append("")
        for fact in facts[:10]:
            lines.append(f"  - {fact['fact']}")
        if len(facts) > 10:
            lines.append(f"  ... and {len(facts) - 10} more facts")
        return "\n".join(lines)

    def _build_output(
        self, status: str, report: Optional[str],
        errors: list, query: str, manifest: CrashRecoveryManifest,
    ) -> dict:
        return {
            "status": status,
            "query": query,
            "report": report,
            "total_errors": len(errors),
            "transient_errors": sum(1 for e in errors if e.is_transient),
            "permanent_errors": sum(1 for e in errors if not e.is_transient),
            "step_summaries": [r.summary() for r in self.step_results],
            "manifest_path": manifest.manifest_path,
        }


# --- Run a single pipeline ---
random.seed(42)
np.random.seed(42)

pipeline = ResearchPipeline(
    pipeline_id="demo_run",
    retriever_failure_rate=0.15,
    extractor_failure_rate=0.15,
    synthesizer_failure_rate=0.05,
    max_retries=2,
)

output = pipeline.run("transformer architectures", n_docs=5)

print(f"Pipeline Status: {output['status']}")
print(f"Errors: {output['total_errors']} ({output['transient_errors']} transient, {output['permanent_errors']} permanent)")
print("\n--- Step Summaries ---")
for summary in output["step_summaries"]:
    print(summary)
    print()

if output["report"]:
    print("--- Report ---")
    print(output["report"][:500])

In [ ]:
#@title 🎧 Listen: Stress Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_stress_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

Let us stress-test the pipeline. We will run it 50 times with injected random failures and measure:
1. **Pipeline success rate** -- how often does the pipeline produce usable output?
2. **Partial result quality** -- when the pipeline partially fails, how much of the output is still usable?
3. **Time saved by crash recovery** -- how much re-computation does crash recovery avoid?

We compare three configurations:
- **No error handling**: pipeline fails completely on any error
- **Retries only**: retry transient failures but no partial results
- **Full system**: retries + partial results + graceful degradation

In [ ]:
# ============================================================
# Stress test: 50 pipeline runs under three configurations
# ============================================================

N_RUNS = 50
RETRIEVER_FAIL = 0.10  # 10% per-document failure
EXTRACTOR_FAIL = 0.10
SYNTHESIZER_FAIL = 0.05

def run_naive_pipeline(query, n_docs=5):
    """Pipeline with NO error handling. Any failure = total failure."""
    retriever = DocumentRetriever(RETRIEVER_FAIL)
    extractor = FactExtractor(EXTRACTOR_FAIL)
    synthesizer = ReportSynthesizer(SYNTHESIZER_FAIL)

    r1 = retriever.retrieve(query, n_docs)
    if r1.status != ResultStatus.SUCCESS:  # Strict: any error = fail
        return {"status": "FAILED", "stage": "retrieve", "facts": 0}

    r2 = extractor.extract(r1.data)
    if r2.status != ResultStatus.SUCCESS:
        return {"status": "FAILED", "stage": "extract", "facts": 0}

    r3 = synthesizer.synthesize(r2.data, query)
    if r3.status != ResultStatus.SUCCESS:
        return {"status": "FAILED", "stage": "synthesize", "facts": 0}

    return {"status": "SUCCESS", "facts": r3.metadata.get("facts_used", 0)}


def run_retry_only_pipeline(query, n_docs=5):
    """Pipeline with retries but no partial results."""
    retriever = DocumentRetriever(RETRIEVER_FAIL)
    extractor = FactExtractor(EXTRACTOR_FAIL)
    synthesizer = ReportSynthesizer(SYNTHESIZER_FAIL)

    # Retry each step up to 2 times, but still require SUCCESS (not partial)
    for attempt in range(3):
        r1 = retriever.retrieve(query, n_docs)
        if r1.status == ResultStatus.SUCCESS:
            break
    if r1.status != ResultStatus.SUCCESS:
        return {"status": "FAILED", "stage": "retrieve", "facts": 0}

    for attempt in range(3):
        r2 = extractor.extract(r1.data)
        if r2.status == ResultStatus.SUCCESS:
            break
    if r2.status != ResultStatus.SUCCESS:
        return {"status": "FAILED", "stage": "extract", "facts": 0}

    for attempt in range(3):
        r3 = synthesizer.synthesize(r2.data, query)
        if r3.status == ResultStatus.SUCCESS:
            break
    if r3.status != ResultStatus.SUCCESS:
        return {"status": "FAILED", "stage": "synthesize", "facts": 0}

    return {"status": "SUCCESS", "facts": r3.metadata.get("facts_used", 0)}


def run_full_pipeline(query, n_docs=5):
    """Pipeline with retries + partial results + graceful degradation."""
    pipeline = ResearchPipeline(
        retriever_failure_rate=RETRIEVER_FAIL,
        extractor_failure_rate=EXTRACTOR_FAIL,
        synthesizer_failure_rate=SYNTHESIZER_FAIL,
        max_retries=2,
    )
    result = pipeline.run(query, n_docs, manifest_dir=tempfile.mkdtemp())
    n_facts = 0
    if result["report"]:
        # Count facts from the step summaries
        for sr in pipeline.step_results:
            if sr.agent_name == "FactExtractor" and sr.data:
                n_facts = len(sr.data)
    return {"status": result["status"], "facts": n_facts}


# Run the experiments
random.seed(42)
np.random.seed(42)

results_naive = []
results_retry = []
results_full = []

print(f"Running {N_RUNS} pipeline executions for each configuration...")
print(f"Failure rates: retriever={RETRIEVER_FAIL*100:.0f}%, extractor={EXTRACTOR_FAIL*100:.0f}%, synthesizer={SYNTHESIZER_FAIL*100:.0f}%")
print()

for i in range(N_RUNS):
    # Use the same seed offset for fair comparison
    random.seed(42 + i * 137)
    results_naive.append(run_naive_pipeline("transformer architectures"))

    random.seed(42 + i * 137)
    results_retry.append(run_retry_only_pipeline("transformer architectures"))

    random.seed(42 + i * 137)
    results_full.append(run_full_pipeline("transformer architectures"))

# Calculate metrics
def calc_metrics(results):
    successes = sum(1 for r in results if r["status"] not in ("FAILED",))
    facts = [r["facts"] for r in results if r["status"] not in ("FAILED",)]
    return {
        "success_rate": successes / len(results) * 100,
        "avg_facts": np.mean(facts) if facts else 0,
        "total_successes": successes,
    }

m_naive = calc_metrics(results_naive)
m_retry = calc_metrics(results_retry)
m_full = calc_metrics(results_full)

print(f"{'Configuration':<25} {'Success Rate':<15} {'Avg Facts':<12} {'Usable Runs':<12}")
print("=" * 65)
print(f"{'No Error Handling':<25} {m_naive['success_rate']:>6.1f}%        {m_naive['avg_facts']:>6.1f}       {m_naive['total_successes']:>3}/{N_RUNS}")
print(f"{'Retries Only':<25} {m_retry['success_rate']:>6.1f}%        {m_retry['avg_facts']:>6.1f}       {m_retry['total_successes']:>3}/{N_RUNS}")
print(f"{'Full System':<25} {m_full['success_rate']:>6.1f}%        {m_full['avg_facts']:>6.1f}       {m_full['total_successes']:>3}/{N_RUNS}")

In [ ]:
# ============================================================
# Visualize the results
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Success Rate Comparison ---
ax = axes[0]
configs = ["No Error\nHandling", "Retries\nOnly", "Full\nSystem"]
rates = [m_naive["success_rate"], m_retry["success_rate"], m_full["success_rate"]]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

bars = ax.bar(configs, rates, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{rate:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_ylabel("Pipeline Success Rate (%)")
ax.set_title(f"Success Rate ({N_RUNS} runs)")
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis="y")

# --- Plot 2: Facts Retrieved Distribution ---
ax = axes[1]
facts_naive = [r["facts"] for r in results_naive]
facts_retry = [r["facts"] for r in results_retry]
facts_full = [r["facts"] for r in results_full]

bp = ax.boxplot(
    [facts_naive, facts_retry, facts_full],
    labels=["No Error\nHandling", "Retries\nOnly", "Full\nSystem"],
    patch_artist=True,
    medianprops={"color": "black", "linewidth": 2},
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Facts Retrieved per Run")
ax.set_title("Output Quality (Facts Extracted)")
ax.grid(True, alpha=0.3, axis="y")

# --- Plot 3: Crash Recovery Time Savings ---
ax = axes[2]
# Simulate crash recovery savings
# For a 3-step pipeline, if crash happens at step i, we save i steps
n_pipeline_steps = 3
simulated_crashes = 20
crash_points = np.random.choice(range(n_pipeline_steps), size=simulated_crashes)

# Cost per step (simulated in relative units)
step_costs = [1.0, 2.0, 1.5]  # Retrieval is cheap, extraction is expensive
total_cost = sum(step_costs)

savings = []
for crash_at in crash_points:
    cost_without = total_cost  # Re-run everything
    cost_with = sum(step_costs[crash_at:])  # Only re-run from crash point
    saving_pct = (1 - cost_with / cost_without) * 100
    savings.append(saving_pct)

ax.hist(savings, bins=15, color="#3498db", alpha=0.8, edgecolor="navy")
ax.axvline(x=np.mean(savings), color="red", linestyle="--", linewidth=2,
           label=f"Mean: {np.mean(savings):.1f}%")
ax.set_xlabel("Computation Saved (%)")
ax.set_ylabel("Number of Crash Events")
ax.set_title(f"Crash Recovery Savings ({simulated_crashes} crashes)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"\nCrash recovery average savings: {np.mean(savings):.1f}% of computation avoided")

In [ ]:
# ============================================================
# Detailed breakdown: where do failures happen?
# ============================================================

# Analyze failure stages for the naive pipeline
failure_stages_naive = {"retrieve": 0, "extract": 0, "synthesize": 0, "success": 0}
for r in results_naive:
    if r["status"] == "FAILED":
        failure_stages_naive[r.get("stage", "unknown")] += 1
    else:
        failure_stages_naive["success"] += 1

# Analyze failure stages for the full pipeline
status_counts_full = {"SUCCESS": 0, "PARTIAL": 0, "DEGRADED": 0, "FAILED": 0}
for r in results_full:
    status_counts_full[r["status"]] = status_counts_full.get(r["status"], 0) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Naive pipeline failure stages
stages = ["Retrieval", "Extraction", "Synthesis", "Success"]
counts = [failure_stages_naive["retrieve"], failure_stages_naive["extract"],
          failure_stages_naive["synthesize"], failure_stages_naive["success"]]
colors_pie = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]

wedges, texts, autotexts = ax1.pie(
    counts, labels=stages, colors=colors_pie, autopct="%1.0f%%",
    startangle=90, textprops={"fontsize": 10}
)
ax1.set_title(f"Naive Pipeline: Where Failures Occur\n({N_RUNS} runs)")

# Full pipeline outcome distribution
outcomes = ["Full\nSuccess", "Partial\nSuccess", "Degraded", "Failed"]
outcome_counts = [status_counts_full.get("SUCCESS", 0), status_counts_full.get("PARTIAL", 0),
                  status_counts_full.get("DEGRADED", 0), status_counts_full.get("FAILED", 0)]
outcome_colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]

bars = ax2.bar(outcomes, outcome_counts, color=outcome_colors, alpha=0.85,
               edgecolor="black", linewidth=0.8)
for bar, count in zip(bars, outcome_counts):
    if count > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(count), ha="center", va="bottom", fontweight="bold", fontsize=11)
ax2.set_ylabel("Number of Runs")
ax2.set_title(f"Full System: Outcome Distribution\n({N_RUNS} runs)")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("Naive pipeline -- even a 10% per-agent failure rate causes significant pipeline failures.")
print("Full system -- partial success and degradation keep most runs usable.")

## 8. Final Output

In [ ]:
# ============================================================
# Final Output: Summary of everything we built
# ============================================================

print("=" * 65)
print("  ERROR PROPAGATION & CRASH RECOVERY -- SUMMARY")
print("=" * 65)
print()
print("Components Built:")
print("  1. Structured Error Result Type (PipelineResult)")
print("     - Three-tier status: SUCCESS / PARTIAL_SUCCESS / FAILURE")
print("     - Full error context: type, severity, transience, source agent")
print("     - Metadata propagation across pipeline stages")
print()
print("  2. Transient vs Permanent Failure Classifier")
print("     - Automatic classification of error types")
print("     - Retries only for transient failures (timeouts, rate limits)")
print("     - Immediate abort for permanent failures (auth, parse errors)")
print()
print("  3. Crash Recovery Manifest")
print("     - JSON-based checkpoint after every pipeline step")
print("     - Resume from last completed step after crash")
print("     - Cached intermediate outputs avoid recomputation")
print()
print("  4. Graceful Degradation")
print("     - Partial results instead of total failure")
print("     - Fallback strategies when late-stage agents fail")
print("     - Quality annotations on degraded outputs")
print()
print("Experimental Results (50 pipeline runs, 10% per-agent failure):")
print(f"  No error handling:  {m_naive['success_rate']:5.1f}% success rate, {m_naive['avg_facts']:.1f} avg facts")
print(f"  Retries only:       {m_retry['success_rate']:5.1f}% success rate, {m_retry['avg_facts']:.1f} avg facts")
print(f"  Full system:        {m_full['success_rate']:5.1f}% success rate, {m_full['avg_facts']:.1f} avg facts")
print()
print("Key Insight:")
print("  Pipeline reliability drops exponentially with the number of agents.")
print("  Structured error propagation + partial results + crash recovery")
print("  transform a fragile pipeline into a resilient system that almost")
print("  always produces usable output -- even when individual agents fail.")
print()
print("=" * 65)

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What We Learned

1. **Silent failures are the enemy.** Multi-agent pipelines that swallow errors produce confident-sounding garbage. Every agent must return a structured result that says exactly what succeeded, what failed, and why.

2. **Not all failures deserve retries.** Classifying errors as transient (timeout, rate limit) vs permanent (invalid input, authentication) prevents wasting time and money retrying unfixable problems.

3. **Partial results are almost always better than no results.** If 4 out of 5 documents were processed successfully, returning those 4 results with a warning is far more useful than returning nothing.

4. **Crash recovery is cheap insurance.** Writing a JSON checkpoint after each step costs almost nothing. Losing hours of computation to a crash costs everything.

5. **Reliability compounds negatively.** A 3-agent pipeline with 95%-reliable agents is only 85.7% reliable. By the time you reach 10 agents, you are below 60%. Error handling is not optional at scale -- it is the difference between a system that works and one that does not.

### Key Equations

| Concept | Formula |
|:--------|:--------|
| Pipeline reliability (no handling) | $P = p^n$ |
| With $k$ retries (transient fraction $t$) | $P_{\text{agent}} = 1 - (1-p) \cdot [(1-t) + t \cdot (1-p)^k]$ |
| Crash recovery savings | $\text{saved} = \sum_{j=0}^{i-1} \text{cost}(\text{step}_j)$ |
| Exponential backoff with jitter | $\text{delay} = \text{rand}(0, \min(\text{cap}, \text{base} \cdot 2^{\text{attempt}}))$ |

### What Is Next

In the next notebook, we will build on these reliability primitives to implement **context window management under failure conditions**. When an agent fails and we need to retry or fall back, how do we manage the context window? How do we ensure that error context, partial results, and recovery metadata all fit within the token budget? That is where context management and reliability intersect.

### Further Reading

- Nygard, *Release It! Design and Deploy Production-Ready Software* (2018) -- the definitive guide to stability patterns
- Amazon Builders' Library, "Timeouts, retries, and backoff with jitter" -- production retry strategies at scale
- Kleppmann, *Designing Data-Intensive Applications* (2017), Ch. 8 -- fault tolerance in distributed systems
- Anthropic, "Building reliable multi-agent systems" (2025) -- error handling patterns for LLM agent pipelines